# Training the autoencoders on a Google Colab GPU

This notebook trains the learned detectors of the thesis, **conv_ae, fc_ae and conv_vae** for every fold and seed
(and, optionally, the latent-size ablations), on a Colab GPU. Everything else stays on the laptop: capture,
preprocessing, PCA, the energy detectors, the PSS correlator, calibration, evaluation, latency and the real-time demo.

The notebook only calls `colab/colab_train.py`, which calls `python -m occupancy train --pack ...`; it holds no
training logic of its own. Nothing here needs the RTL-SDR or `pyrtlsdr`.

**Before you start, on the laptop** (in `code/`):
1. `uv run python -m occupancy preprocess`
2. `uv run python -m occupancy pack` writes `work/colab/train_pack.zip` and `work/colab/occupancy_code.zip`.
3. Upload both files to Google Drive, into the folder `MyDrive/msc-thesis/`.

**Here:** *Runtime > Change runtime type >* a GPU (for example T4), then run the cells from top to bottom.

**Disconnects.** Every run writes a checkpoint to `MyDrive/msc-thesis/runs/` after every epoch. If the runtime
disconnects, reconnect and run all cells again (the timing cell, step 6, can be skipped): finished runs are
skipped and the unfinished run continues from its last checkpoint.

**Reproducibility.** The seeds fix the initial weights and the order of the training frames, but GPU training is
not guaranteed to be bit-identical between repetitions: the backward pass of bilinear upsampling (in the conv_ae and
conv_vae decoders) has no deterministic CUDA kernel. Every run records the GPU, the CUDA, cuDNN and PyTorch versions,
the determinism settings and the code version in its `train_summary.json`.

**At the end**, download `MyDrive/msc-thesis/runs.zip` and import it on the laptop (step 10).

## 1. Settings
All paths are relative to `MyDrive` and set only here. Change them only if you uploaded under other names.

In [ ]:
# --- Settings: the only cell you may need to edit --------------------------------------------
DRIVE_FOLDER = "msc-thesis"          # folder in MyDrive with the uploads and, later, the outputs
CODE_ZIP = "occupancy_code.zip"      # written by `python -m occupancy pack`
PACK_ZIP = "train_pack.zip"          # written by `python -m occupancy pack` (train_pack_SYNTHETIC.zip = rehearsal)
RUNS = "runs"                        # output folder inside DRIVE_FOLDER: checkpoints, histories, models
RUNS_ZIP = "runs.zip"                # the file to download at the end
TRAIN_ABLATIONS = True               # after the main models, also train the latent-size ablations
UNITS_PER_HOUR = None                # compute units per hour of this runtime (shown by Colab); None = not converted
ALLOW_SYNTHETIC = False              # True only to rehearse the workflow with a SYNTHETIC pack

MOUNT = "/content/drive"
DRIVE = f"{MOUNT}/MyDrive/{DRIVE_FOLDER}"
LOCAL = "/content/msc-thesis"        # local disk of the runtime: fast, but emptied by every disconnect
CODE_DIR = f"{LOCAL}/occupancy_code"
SCRIPT = f"{CODE_DIR}/colab/colab_train.py"
PACK = f"{LOCAL}/{PACK_ZIP[:-len('.zip')]}"
OUT = f"{DRIVE}/{RUNS}"
SYN = "--allow-synthetic" if ALLOW_SYNTHETIC else ""
ABL = "--ablation" if TRAIN_ABLATIONS else ""
UNITS = f"--units-per-hour {UNITS_PER_HOUR}" if UNITS_PER_HOUR else ""
print("Drive folder:", DRIVE)
print("outputs:     ", OUT)

## 2. Mount Google Drive
Colab asks for permission to access your Drive. The cell then checks that both uploads are there.

In [ ]:
from google.colab import drive
drive.mount(MOUNT)

import os
for name in (CODE_ZIP, PACK_ZIP):
    path = f"{DRIVE}/{name}"
    print(("found    " if os.path.isfile(path) else "MISSING  ") + path)

## 3. Check the GPU
`nvidia-smi` shows the GPU Colab assigned, and PyTorch must see it. The check runs in a separate process, so the
notebook itself does not hold GPU memory. If there is no GPU: *Runtime > Change runtime type > GPU*, then start again.

In [ ]:
!nvidia-smi
!python -c "import torch; print('PyTorch', torch.__version__, '| CUDA build', torch.version.cuda, '| GPU available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')"

## 4. Copy the code and the training pack to the runtime
The code is unzipped to the runtime's local disk. The pack zip is copied from Drive to the local disk (training
reads it many times, and the local disk is much faster than Drive), unzipped, and every file is checked against its
SHA-256 checksum in the pack manifest. If the same verified pack is already there, nothing is copied.

In [ ]:
import os, shutil, zipfile
os.makedirs(LOCAL, exist_ok=True)
shutil.rmtree(CODE_DIR, ignore_errors=True)
with zipfile.ZipFile(f"{DRIVE}/{CODE_ZIP}") as z:      # your own code archive, written by `occupancy pack`
    z.extractall(LOCAL)
import json
print("code version:", json.load(open(f"{CODE_DIR}/code_version.json"))["code_sha256"][:12])
!python "{SCRIPT}" setup --pack-zip "{DRIVE}/{PACK_ZIP}" --local "{LOCAL}"

## 5. Python packages
Colab already has NumPy, SciPy and PyTorch with CUDA. The check lists what is missing and installs only that.
PyTorch is never reinstalled here (that would replace Colab's CUDA build).

In [ ]:
import json, subprocess, sys
env = json.loads(subprocess.run([sys.executable, SCRIPT, "check-env", "--json"],
                                capture_output=True, text=True, check=True).stdout)
print("python", env["python"], env["versions"])
print("GPU:", env["gpu"])
if "torch" in env["missing"]:
    raise SystemExit("PyTorch is missing: use a standard Colab GPU runtime (do not pip-install torch here)")
missing = " ".join(p for p in env["missing"] if p != "torch")
if missing:
    print("installing:", missing)
    %pip install -q {missing}
else:
    print("nothing to install")

## 6. Time one epoch per model (once, a few minutes)
Times one epoch of conv_ae, fc_ae and conv_vae on fold 0 on this GPU, and one checkpoint write to Drive. It then
estimates the hours for the whole design (every model, fold and seed, with the ablations) at 30, 60 and 100 epochs
per run; early stopping ends most runs before 100. With `UNITS_PER_HOUR` set, the hours are also converted to compute
units (Colab bills the time the runtime is connected). It also shows how much checkpoint data training will write to
Drive (an fc_ae checkpoint is about 136 MB, written after every epoch). The result is saved as `runs/time_epoch.json`.

In [ ]:
!python "{SCRIPT}" time-epoch --pack "{PACK}" --out "{OUT}" {UNITS} {SYN}

## 7. Train (re-run this cell after a disconnect)
Trains conv_ae, fc_ae and conv_vae for every fold and seed, then, if `TRAIN_ABLATIONS` is set, the latent-size
ablations. Each run writes a checkpoint to Drive after every epoch and a completion marker when it has finished.
Finished runs are skipped and an unfinished run continues where it stopped, so after a disconnect: reconnect, run
steps 1 to 5 again, then this cell. A failed attempt (for example a Drive input/output error) is retried
automatically; a refusal (wrong pack, settings or code) is not. The output is also written to `runs/logs/`.
Run only one training notebook at a time on the same `runs` folder.

In [ ]:
!python "{SCRIPT}" train --pack "{PACK}" --out "{OUT}" {ABL} {SYN} --max-attempts 5

## 8. Progress
Counts complete, unfinished and not-started runs. It can be run at any time.

In [ ]:
!python "{SCRIPT}" status --pack "{PACK}" --out "{OUT}" {ABL}

## 9. Zip the outputs for download
Writes `runs.zip` next to the `runs` folder: models, histories, summaries, completion markers, the timing and the
logs. Checkpoints are left out (finished runs have none, and unfinished runs are not imported). Then download
`MyDrive/msc-thesis/runs.zip` from Google Drive.

In [ ]:
!python "{SCRIPT}" zip-runs --out "{OUT}" --zip "{DRIVE}/{RUNS_ZIP}"

Optional, when you are done with this runtime: write everything to Drive and unmount it.

In [ ]:
drive.flush_and_unmount()

## 10. Back on the laptop (in `code/`)
```bash
uv run python -m occupancy unpack --runs ~/Downloads/runs.zip    # verify and import the trained models
uv run python -m occupancy train --models pca                    # PCA is fitted on the laptop
uv run python -m occupancy calibrate --ablation                  # thresholds (omit --ablation if not trained)
uv run python -m occupancy evaluate --ablation
uv run python -m occupancy latency
```
`unpack` refuses runs from another pack, other training settings or other code (unless `--allow-code-change`, which
is recorded), and lists every expected run that is missing or unfinished. Its report is `work/colab_import.json`.